# Classwork: Curve of Best Fit

**Goal:** Use Mean Squared Error (MSE) to decide which polynomial (line, parabola, cubic, ...) best fits a set of data, and find out why the curve with the *lowest* error is not always the best choice.

**Read first:** The **Numpy Polyfit** section and its *Comments* tab from[Curve of Best Fit](../visualizations/curve_of_best_fit.md). You don't need to read all of this right now. Save the *Choosing the Degree with MSE* section for after Part 3.

| Part | What you do |
|---|---|
| 1 | Compare a line and a parabola using RMSE |
| 2 | Fit polynomials of degree 1 to 10 to mystery data and compare their RMSE |
| 3 | Test those same curves on *new* data and see which one really predicts best |

**RMSE** is the square root of the MSE: `np.sqrt(mse)`. It is in the same units as `y`, so it is easier to read: "the curve is off by about this much, on average."

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error

## Part 1: Line vs. parabola

The code below makes noisy points from a parabola and plots them. The **parabola** work is done for you:
- `sns.regplot(order=2)` draws the best-fit parabola (red)
- `np.polyfit(x_data, y_data, 2)` finds the parabola's coefficients `[a, b, c]`
- the coefficients give a predicted y for every x, and `mean_squared_error` compares those to the real y values

**Your task:** do the same for a **line** (degree 1) in the `TODO` section:
1. Use `np.polyfit` to get the line's coefficients: `[slope, intercept]`
2. Replace the placeholder `y_predicted_1` with the line's predicted y for every x in `x_data`
3. Replace the placeholder `mse_degree_1 = 0` with the line's MSE

**You're done when** the green line runs through the middle of the points, and the RMSE box shows a nonzero value for `Line`.

In [ ]:
def add_inset_text(fig, position, actual, coeffs):
    """Add a text box to fig at position [x, y, width, height] (figure fractions)
    comparing the actual coefficients to the predicted ones."""
    ax_inset = fig.add_axes(position)
    ax_inset.set_xticks([])
    ax_inset.set_yticks([])
    ax_inset.grid(False)
    ax_inset.text(0.05, 0.90, 'Coeff Actual vs Pred', ha='left', va='top', fontsize=14)
    text = ''
    for a, c in zip(actual, coeffs):
        text += f'{a:.2f} vs {c:.2f}\n'
    ax_inset.text(0.18, 0.70, text, ha='left', va='top', fontsize=12)
    return ax_inset


def parabola(x, a, b, c):
    return a*x**2 + b*x + c


def generate_points(a, b, c, noise):
    x_data = [x for x in range(3, 50, 2)]
    y_data = [parabola(x, a, b, c) + random.randint(-noise, noise) for x in x_data]
    return x_data, y_data


def best_parabola_fit():
    # the "true" parabola coefficients that the noisy points come from
    actual = (0.6, 5, 1)
    x_data, y_data = generate_points(*actual, noise=150)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.grid(True, linestyle='--', linewidth=0.5, color='black')
    # scatter the points and draw the best-fit parabola in red, with no confidence shading
    sns.regplot(x=x_data, y=y_data, ax=ax, order=2, ci=None, line_kws={'color': 'red'})

    # ---- Parabola (degree 2): done for you ----
    # coefficients are ordered from highest power to lowest: [a, b, c]
    coeffs_2 = np.polyfit(x_data, y_data, 2)
    y_predicted_2 = [coeffs_2[0]*x**2 + coeffs_2[1]*x + coeffs_2[2] for x in x_data]
    mse_degree_2 = mean_squared_error(y_data, y_predicted_2)

    # ---- Line (degree 1): TODO ----
    # 1. coeffs_1 = np.polyfit(...)   ->  [slope, intercept]
    # 2. replace the placeholder below with the line's predicted y values
    # 3. replace 0 with the line's MSE
    y_predicted_1 = [30*x + 1 for x in x_data]   # PLACEHOLDER: replace
    mse_degree_1 = 0                             # PLACEHOLDER: replace

    print('RMSE Line:', np.sqrt(mse_degree_1), '  RMSE Parabola:', np.sqrt(mse_degree_2))

    # text box with the coefficients and RMSE values
    ax_inset = add_inset_text(fig, [0.15, 0.50, 0.35, 0.28], actual, coeffs_2)
    ax_inset.text(0.7, 0.90, 'RMSE:', ha='left', va='top', fontsize=14)
    text = f'Line:{np.sqrt(mse_degree_1):.0f}\nPara:{np.sqrt(mse_degree_2):.0f}'
    ax_inset.text(0.65, 0.70, text, ha='left', va='top', fontsize=12)

    # use ax.plot (not plt.plot) so the line is drawn on the main plot, not the inset
    ax.plot(x_data, y_predicted_1, color='green')
    ax.set_title('Best Parabola & Line Fit', fontsize=16)
    ax.legend(['Actual', 'Parabola Fit', 'Line Fit'], loc='lower right')


best_parabola_fit()

**Question 1:** Which fits these points better, the line or the parabola? Which RMSE numbers tell you that?

*Your answer here*

## Part 2: More is better?

If a parabola beats a line, would a cubic beat the parabola? Would degree 10 beat them all?

`best-fit-mystery.csv` has data from a mystery experiment. Each row is marked `train` or `future`:
- **train**: the 12 points we measured. Fit your curves to these.
- **future**: points measured *later*. **Don't use these until Part 3.**

In [ ]:
df = pd.read_csv('best-fit-mystery.csv')
train = df[df['split'] == 'train']
future = df[df['split'] == 'future']

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(train['x'], train['y'])
ax.grid(True, linestyle='--', linewidth=0.5)
ax.set_title('Mystery data: training points', fontsize=16)

**Your task:**
1. Finish `rmse(coeffs, data)`. It returns how far the polynomial with those coefficients is from the points in `data`.
   `np.polyval(coeffs, xs)` evaluates the polynomial at every x in `xs`. It does what the list comprehension did in Part 1, for any degree.
2. Finish the loop. For each degree from 1 to 10, fit a polynomial to the **training** points and add its RMSE **on the training points** to `train_rmse`.

In [ ]:
def rmse(coeffs, data):
    """Return the RMSE between the polynomial with these coefficients and data's points."""
    # TODO: predict y for every x in data['x'], then compare to data['y']
    return 0


degrees = list(range(1, 11))
train_rmse = []
for degree in degrees:
    # TODO: fit a polynomial of this degree to the training points,
    #       then append its RMSE on the training points to train_rmse
    pass

for degree, error in zip(degrees, train_rmse):
    print(f'degree {degree:2}: training RMSE = {error:8.1f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(degrees, train_rmse, marker='o', label='training points')
ax.set_xticks(degrees)
ax.set_xlabel('Polynomial degree')
ax.set_ylabel('RMSE')
ax.grid(True, linestyle='--', linewidth=0.5)
ax.legend()
ax.set_title('RMSE vs. degree', fontsize=16)

**Question 2:** What happens to the training RMSE as the degree goes up? Which degree has the lowest?

**Question 3 (prediction):** Before you start Part 3, which degree would you trust most to predict *new* points from this experiment? Why?

*Your answer here*

## Part 3: Test it on new data

A curve of best fit is only useful if it **predicts** points it has never seen. Now use the `future` points.

**Your task:**
1. Finish the loop. Fit each degree to the **training** points again (as in Part 2), but compute its RMSE **on the future points**.
2. Run the plot cell. The y-axis uses a log scale, because some errors are huge.
3. Set `best_degree` to the degree with the lowest future RMSE, and run the last cell to compare it with degree 10.

In [ ]:
future_rmse = []
for degree in degrees:
    # TODO: fit to the TRAINING points, then append the RMSE on the FUTURE points
    pass

for degree, tr, fu in zip(degrees, train_rmse, future_rmse):
    print(f'degree {degree:2}: training RMSE = {tr:8.1f}   future RMSE = {fu:10.1f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(degrees, train_rmse, marker='o', label='training points')
ax.plot(degrees, future_rmse, marker='s', label='future points')
ax.set_yscale('log')
ax.set_xticks(degrees)
ax.set_xlabel('Polynomial degree')
ax.set_ylabel('RMSE (log scale)')
ax.grid(True, linestyle='--', linewidth=0.5)
ax.legend()
ax.set_title('Training vs. future RMSE', fontsize=16)

In [ ]:
best_degree = 1   # TODO: the degree with the lowest future RMSE

xs = np.linspace(df['x'].min(), df['x'].max(), 400)
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(train['x'], train['y'], label='training points')
ax.scatter(future['x'], future['y'], marker='x', color='gray', label='future points')
for degree, color in [(best_degree, 'green'), (10, 'red')]:
    coeffs = np.polyfit(train['x'], train['y'], degree)
    ax.plot(xs, np.polyval(coeffs, xs), color=color, label=f'degree {degree}')
ax.axvline(train['x'].max(), linestyle='--', color='black')
ax.set_ylim(df['y'].min() - 100, df['y'].max() + 100)
ax.grid(True, linestyle='--', linewidth=0.5)
ax.legend()
ax.set_title(f'Degree {best_degree} vs. degree 10', fontsize=16)

**Question 4:** Which degree predicted the future points best? Was your prediction in Question 3 right?

**Question 5:** Degree 10 had the lowest *training* RMSE but terrible *future* RMSE. Use the last plot to explain why.

**When you're done:** check your thinking against [Choosing the Degree with MSE](../visualizations/curve_of_best_fit.md#choosing-the-degree-with-mse) on the reference page.

*Your answer here*